# Spatial Experiment — Target Detection via Score Matching

**Section 5**: CF-Attn + NeighborMLP vs THANTD on Pavia-U hyperspectral data.

This notebook:
1. Mounts Google Drive
2. Copies `pythonProject` to `/content/proj`
3. Runs the full experiment (8 scenarios × 3 box sizes)
4. Displays all figures inline

**Before running**: upload `pythonProject.zip` to `/MyDrive/final_paper/`

In [ ]:
# ── Cell 1: Mount Drive & install deps ────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys

# Install any missing packages
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'scikit-learn', 'scipy', 'tqdm', 'matplotlib', 'pyyaml'],
               check=True)
print('Dependencies OK')

In [ ]:
# ── Cell 2: Copy project from Drive ───────────────────────────────────────
import os, shutil, sys

DRIVE_PROJECT = '/content/drive/MyDrive/final_paper/pythonProject'
LOCAL_PROJECT = '/content/proj'

if os.path.exists(DRIVE_PROJECT):
    if os.path.exists(LOCAL_PROJECT):
        shutil.rmtree(LOCAL_PROJECT)
    shutil.copytree(DRIVE_PROJECT, LOCAL_PROJECT)
    print(f'Copied {DRIVE_PROJECT} → {LOCAL_PROJECT}')
else:
    # Try zip
    zip_path = '/content/drive/MyDrive/final_paper/pythonProject.zip'
    if os.path.exists(zip_path):
        import zipfile
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall('/content')
        os.rename('/content/pythonProject', LOCAL_PROJECT)
        print(f'Extracted {zip_path} → {LOCAL_PROJECT}')
    else:
        raise FileNotFoundError(
            f'Could not find project at {DRIVE_PROJECT} or {zip_path}\n'
            f'Please upload pythonProject (or pythonProject.zip) to '
            f'/MyDrive/final_paper/ on Google Drive.')

# Add project root to Python path
if LOCAL_PROJECT not in sys.path:
    sys.path.insert(0, LOCAL_PROJECT)
# Also add the spatial experiments directory
SPATIAL_DIR = os.path.join(LOCAL_PROJECT, 'experiments', 'spatial')
if SPATIAL_DIR not in sys.path:
    sys.path.insert(0, SPATIAL_DIR)

os.chdir(LOCAL_PROJECT)
print(f'CWD: {os.getcwd()}')
print('Project structure:')
for f in sorted(os.listdir(LOCAL_PROJECT))[:20]:
    print(f'  {f}')

In [ ]:
# ── Cell 3a: DRY RUN first — verify all outputs are correct ──────────────
# This runs 1 scenario with 10 epochs each model.
# Check all outputs PASS before the full run.
import subprocess, sys

RESULTS_DIR = '/content/drive/MyDrive/final_paper/spatial_results'

result = subprocess.run(
    [sys.executable, '-u',
     'experiments/spatial/run_colab.py',
     '--config', 'experiments/spatial/colab.yaml',
     '--results_dir', RESULTS_DIR,
     '--dry-run',
     '--no-thantd'],    # remove --no-thantd if GPU is available
    capture_output=False,
    text=True,
)
print('Exit code:', result.returncode)
if result.returncode != 0:
    print('DRY RUN FAILED — do not proceed to full run until all checks pass.')
else:
    print('DRY RUN PASSED — proceed to Cell 3b for full run.')

In [ ]:
# ── Cell 3b: FULL RUN ─────────────────────────────────────────────────────
# ~2-3 hours on T4 with all 8 scenarios × 3 box sizes.
# Checkpoints are saved after each model — safe to restart on disconnect.
import subprocess, sys

RESULTS_DIR = '/content/drive/MyDrive/final_paper/spatial_results'

result = subprocess.run(
    [sys.executable, '-u',
     'experiments/spatial/run_colab.py',
     '--config', 'experiments/spatial/colab.yaml',
     '--results_dir', RESULTS_DIR,
     # '--no-thantd',    # uncomment if THANTD is too slow
    ],
    capture_output=False,
    text=True,
)
print('Exit code:', result.returncode)

In [ ]:
# ── Cell 4: Display all figures inline ───────────────────────────────────
import os, glob, json
from IPython.display import Image, display, HTML
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

RESULTS_DIR = '/content/drive/MyDrive/final_paper/spatial_results'

# Find the latest run
runs = sorted(glob.glob(os.path.join(RESULTS_DIR, 'colab_*')))
if not runs:
    print('No results found. Run Cell 3 first.')
else:
    run_dir = runs[-1]
    print(f'Showing results from: {run_dir}')

    # Print AUC summary from all_metrics.json
    metrics_path = os.path.join(run_dir, 'all_metrics.json')
    if os.path.exists(metrics_path):
        with open(metrics_path) as f:
            all_metrics = json.load(f)
        print('\n=== AUC Summary ===')
        for sid_key, sid_data in all_metrics.items():
            for bkey, bdata in sid_data.items():
                if not bkey.startswith('n'):
                    continue
                print(f'\n{sid_key} {bkey} — additive AUC:')
                add_auc = bdata.get('additive', {}).get('auc', {})
                for nm, v in add_auc.items():
                    if isinstance(v, float):
                        print(f'  {nm:20s}: {v:.4f}')

    # Show aggregated figures
    agg_figs = sorted(glob.glob(os.path.join(run_dir, 'figures', '*.pdf')))
    print(f'\nAggregated figures ({len(agg_figs)}):')
    for f in agg_figs:
        print(f'  {os.path.basename(f)}')

    # Convert PDFs to PNG and display (requires poppler)
    try:
        import subprocess as sp
        sp.run(['apt-get', 'install', '-y', '-q', 'poppler-utils'], check=True)
        for pdf in agg_figs[:5]:   # show first 5 aggregated figures
            png = pdf.replace('.pdf', '.png')
            sp.run(['pdftoppm', '-r', '150', '-png', '-singlefile', pdf,
                    png.replace('.png', '')], check=True)
            if os.path.exists(png):
                img = mpimg.imread(png)
                fig, ax = plt.subplots(figsize=(14, 8))
                ax.imshow(img); ax.axis('off')
                ax.set_title(os.path.basename(pdf))
                plt.tight_layout(); plt.show()
    except Exception as e:
        print(f'PDF display error: {e}')
        print('PDFs saved to Drive — view them there.')